In [1]:
import duckdb
import numpy as np
import pandas as pd
from faultlines.pipeline import get_log_return_matrix
import networkx as nx
from sklearn.covariance import GraphicalLassoCV

# Connect to your local database
conn = duckdb.connect("faultlines.duckdb")

# Pull log returns for your target universe (e.g., top stocks for 2016)
returns_df = get_log_return_matrix(
    "2016-01-01", "2016-12-31", asset_type="Stock"
)
print(returns_df.head())

Ticker             a        aa      aaap      aaba       aac       aal  \
Date                                                                     
2016-01-04 -0.027143 -0.016525 -0.019374 -0.057548 -0.028741 -0.034597   
2016-01-05 -0.003433 -0.046317  0.006824  0.025159  0.049510 -0.009544   
2016-01-06  0.004408 -0.073844 -0.030246 -0.001243 -0.023396  0.017367   
2016-01-07 -0.043436 -0.040336 -0.010401 -0.064207 -0.043544 -0.019130   
2016-01-08 -0.010520 -0.024477 -0.057258  0.015463 -0.004957 -0.001994   

Ticker          aamc       aan      aaoi      aaon  ...      ziop      zixi  \
Date                                                ...                       
2016-01-04  0.153651  0.012816 -0.052030 -0.047143  ... -0.041775 -0.052537   
2016-01-05 -0.004508  0.019205 -0.012975 -0.002240  ...  0.001254  0.010320   
2016-01-06  0.002507 -0.078665 -0.001867 -0.018758  ... -0.091808 -0.018653   
2016-01-07 -0.254682 -0.033372 -0.048510 -0.022402  ... -0.023629  0.000000   
2016-01

In [5]:
import warnings
import duckdb
import networkx as nx
import numpy as np
import pandas as pd
from faultlines.pipeline import get_log_return_matrix
from sklearn.covariance import GraphicalLassoCV
from sklearn.preprocessing import StandardScaler

# Suppress NumPy runtime warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning)

# 1. Load log return matrix from DuckDB
print("Loading log return matrix from DuckDB...")
returns_df = get_log_return_matrix("2016-01-01", "2016-12-31", asset_type="Stock")

# Drop columns with any missing values across the 2016 period, pick top 50
valid_cols = returns_df.dropna(axis=1, how='any')
returns_subset = valid_cols.iloc[:, :50]

# Standardize data explicitly: mean=0, variance=1 (removes scaling issues and constant columns)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(returns_subset)
returns_clean = pd.DataFrame(scaled_data, index=returns_subset.index, columns=returns_subset.columns)

print(f"Cleaned and standardized dataset shape: {returns_clean.shape}")

# 2. Fit Graphical Lasso with Cross-Validation
print("Fitting Graphical Lasso CV...")
model = GraphicalLassoCV(cv=5, n_jobs=-1)
model.fit(returns_clean)

# Extract precision matrix (inverse covariance)
precision = model.precision_

# 3. Convert Precision Matrix to Partial Correlation Matrix
inv_std = np.sqrt(np.maximum(1e-8, np.diagonal(precision)))
partial_corr = -precision / np.outer(inv_std, inv_std)
np.fill_diagonal(partial_corr, 1.0)

pcorr_df = pd.DataFrame(partial_corr, index=returns_clean.columns, columns=returns_clean.columns)
print("Partial correlation matrix shape:", pcorr_df.shape)

# 4. Build Distance Matrix and Network Backbones
tickers = pcorr_df.columns.tolist()
corr_matrix = pcorr_df.values

# Metric distance: d_ij = sqrt(2 * (1 - rho_ij_partial))
dist_matrix = np.sqrt(np.maximum(0, 2 * (1 - corr_matrix)))

# Construct complete graph
G = nx.Graph()
for i in range(len(tickers)):
  for j in range(i + 1, len(tickers)):
    weight = float(dist_matrix[i, j])
    G.add_edge(tickers[i], tickers[j], weight=weight, distance=weight)

# Extract Minimum Spanning Tree (MST)
print("Extracting Minimum Spanning Tree (MST)...")
mst = nx.minimum_spanning_tree(G, weight="weight")
print(f"MST built successfully with {mst.number_of_edges()} edges across {mst.number_of_nodes()} nodes.")

Loading log return matrix from DuckDB...
Cleaned and standardized dataset shape: (270, 50)
Fitting Graphical Lasso CV...
Partial correlation matrix shape: (50, 50)
Extracting Minimum Spanning Tree (MST)...
MST built successfully with 49 edges across 50 nodes.


In [6]:
import os
import warnings
import duckdb
import networkx as nx
import numpy as np
import pandas as pd
from faultlines.pipeline import get_log_return_matrix, get_ticker_metadata
from sklearn.covariance import GraphicalLassoCV
from sklearn.preprocessing import StandardScaler

# Suppress warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# 1. Ensure results/task3 directory exists
os.makedirs("results/task3", exist_ok=True)

# 2. Load log return matrix from DuckDB
print("Loading log return matrix from DuckDB...")
returns_df = get_log_return_matrix("2016-01-01", "2016-12-31", asset_type="Stock")

# Clean data: drop columns with any missing values across the period, pick top 50
valid_cols = returns_df.dropna(axis=1, how='any')
returns_subset = valid_cols.iloc[:, :50]

# Standardize data explicitly
scaler = StandardScaler()
scaled_data = scaler.fit_transform(returns_subset)
returns_clean = pd.DataFrame(
    scaled_data, index=returns_subset.index, columns=returns_subset.columns
)

print(f"Cleaned and standardized dataset shape: {returns_clean.shape}")

# 3. Fit Graphical Lasso & Compute Partial Correlations
print("Fitting Graphical Lasso CV...")
model = GraphicalLassoCV(cv=5, n_jobs=-1)
model.fit(returns_clean)

precision = model.precision_
inv_std = np.sqrt(np.maximum(1e-8, np.diagonal(precision)))
partial_corr = -precision / np.outer(inv_std, inv_std)
np.fill_diagonal(partial_corr, 1.0)

pcorr_df = pd.DataFrame(
    partial_corr, index=returns_clean.columns, columns=returns_clean.columns
)

# 4. Build Complete Distance Graph & MST
tickers = pcorr_df.columns.tolist()
corr_matrix = pcorr_df.values
dist_matrix = np.sqrt(np.maximum(0, 2 * (1 - corr_matrix)))

G = nx.Graph()
for i in range(len(tickers)):
  for j in range(i + 1, len(tickers)):
    weight = float(dist_matrix[i, j])
    G.add_edge(tickers[i], tickers[j], weight=weight, distance=weight)

print("Extracting Minimum Spanning Tree (MST)...")
mst = nx.minimum_spanning_tree(G, weight="weight")

# 5. Build Planar Maximally Filtered Graph (PMFG)
print("Constructing Planar Maximally Filtered Graph (PMFG)...")
# Sort all edges by ascending distance weight
sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]["weight"])

pmfg = nx.Graph()
pmfg.add_nodes_from(G.nodes())

# PMFG adds edges in greedy order of weight as long as planarity is preserved
for u, v, data in sorted_edges:
  pmfg.add_edge(u, v, **data)
  if not nx.check_planarity(pmfg)[0]:
    pmfg.remove_edge(u, v)
  if pmfg.number_of_edges() == 3 * pmfg.number_of_nodes() - 6:
    break  # Maximum planar graph edge bound reached

print(
    f"PMFG built successfully with {pmfg.number_of_edges()} edges (Max allowed:"
    f" {3 * pmfg.number_of_nodes() - 6})."
)

# 6. Compute Centrality Metrics for MST and PMFG
def get_centrality_df(graph, name):
  deg = nx.degree_centrality(graph)
  bet = nx.betweenness_centrality(graph, weight="weight")
  clo = nx.closeness_centrality(graph, distance="weight")
  df = pd.DataFrame({
      "ticker": list(graph.nodes()),
      f"degree_centrality_{name}": list(deg.values()),
      f"betweenness_centrality_{name}": list(bet.values()),
      f"closeness_centrality_{name}": list(clo.values()),
  })
  return df

mst_cent = get_centrality_df(mst, "mst")
pmfg_cent = get_centrality_df(pmfg, "pmfg")

# Merge centralities together
centrality_df = pd.merge(mst_cent, pmfg_cent, on="ticker")

# 7. Fetch SEC Metadata Sectors and Merge
print("Fetching SEC metadata for tickers...")
sec_meta = get_ticker_metadata(tickers)
if not sec_meta.empty and "ticker" in sec_meta.columns:
  centrality_df = pd.merge(centrality_df, sec_meta, on="ticker", how="left")

# Sort by PMFG Degree Centrality
centrality_df = centrality_df.sort_values(
    by="degree_centrality_pmfg", ascending=False
)

# 8. Export Results to CSV
output_csv_path = "results/task3/network_centrality_summary.csv"
centrality_df.to_csv(output_csv_path, index=False)
print(f"Results successfully saved to {output_csv_path}")

# Display top 10 central stocks in your notebook
display(centrality_df.head(10))

Loading log return matrix from DuckDB...
Cleaned and standardized dataset shape: (270, 50)
Fitting Graphical Lasso CV...
Extracting Minimum Spanning Tree (MST)...
Constructing Planar Maximally Filtered Graph (PMFG)...
PMFG built successfully with 144 edges (Max allowed: 144).
Fetching SEC metadata for tickers...
Results successfully saved to results/task3/network_centrality_summary.csv


,ticker,degree_centrality_mst,betweenness_centrality_mst,closeness_centrality_mst,degree_centrality_pmfg,betweenness_centrality_pmfg,closeness_centrality_pmfg
0,a,0.081633,0.504252,0.176612,0.367347,0.348639,0.403523
21,abbv,0.081633,0.295918,0.137078,0.265306,0.187075,0.361062
17,ab,0.081633,0.193878,0.153899,0.224490,0.183673,0.318143
5,aal,0.040816,0.040816,0.155574,0.224490,0.144558,0.382999
30,abg,0.081633,0.351190,0.143884,0.224490,0.089286,0.352314
43,acad,0.040816,0.153061,0.148270,0.204082,0.105442,0.320134
23,abcb,0.061224,0.119048,0.099114,0.204082,0.116497,0.308507
3,aaba,0.081633,0.599490,0.191237,0.183673,0.116497,0.294642
20,abb,0.061224,0.535714,0.181976,0.183673,0.132653,0.347167
28,abeo,0.020408,0.000000,0.108755,0.163265,0.002551,0.261119
